# Data Quality Gates — `customers_raw`

Runs the five Pandera data-quality gates from `src/telco_churn/data/` against the live
`customers_raw` Postgres table and renders any failures for interactive inspection.

| Gate | Check | Severity |
|---|---|---|
| 1 | Schema — column presence, types, value ranges, categoricals | ERROR |
| 2 | Duplicate `customerid` values | ERROR |
| 3 | Binary churn labels, no missing values | ERROR |
| 4 | Unexpected NULL `totalcharges` (non-zero tenure) | WARNING |
| 5 | Row count ≥ 1 000; critical-column null rates ≤ 5 % | ERROR / WARNING |

**Requires:** `docker compose --profile infra up -d` and `POSTGRES_URL` in `.env`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from telco_churn.data.validate import (
    clean_dataframe,
    validate_clean,
    validate_raw,
)
from telco_churn.utils.db import get_engine
from telco_churn.utils.logging import configure_logging

load_dotenv()
configure_logging()

REPORTS_DIR = Path("reports/validation")

## 1. Load data from Postgres

In [ ]:
engine = get_engine()
df_raw = pd.read_sql("SELECT * FROM customers_raw", engine)
print(f"Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns")
df_raw.head()

## 2. Run the five data-quality gates

In [ ]:
result = validate_raw(df_raw, strict=False, reports_dir=REPORTS_DIR)

## 3. Gate results

In [ ]:
status = "PASS — pipeline can proceed" if result.can_proceed else "FAIL — blocking errors found"
print(f"Overall: {status}")
print(f"Errors  : {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")

gate_summary = pd.DataFrame(
    [
        {
            "gate": c.name,
            "passed": c.passed,
            "severity": str(c.failure_severity),
            "affected_rows": c.affected_rows,
            "message": c.message,
        }
        for c in result.checks
    ]
)
display(gate_summary)

### Failure details

The cells below mirror the `summary.csv` and `schema_failures.csv` written to
`reports/validation/` by `validate_raw`. They render as `None` when all gates pass.

In [ ]:
# summary.csv equivalent — failing checks only
failing = [c for c in result.checks if not c.passed]

if not failing:
    print("summary.csv: no failing checks — file not written.")
else:
    summary_csv = pd.DataFrame(
        [
            {
                "check": c.name,
                "failure_severity": str(c.failure_severity),
                "message": c.message,
                "affected_rows": c.affected_rows,
            }
            for c in failing
        ]
    )
    print("summary.csv")
    display(summary_csv)

In [ ]:
# schema_failures.csv equivalent — row-level pandera failure cases
schema_check = next((c for c in result.checks if c.name == "schema"), None)

if schema_check is None or schema_check.passed:
    print("schema_failures.csv: schema check passed — file not written.")
else:
    print("schema_failures.csv")
    display(schema_check.detail)

## 4. Impute `totalcharges` and re-validate

`clean_dataframe` fills NULL `totalcharges` for the 11 zero-tenure customers with the
column median. `validate_clean` re-runs all gates using `CleanedSchema`, which requires
`totalcharges` to be non-null.

In [ ]:
df_clean = clean_dataframe(df_raw)

nulls_before = int(df_raw["totalcharges"].isna().sum())
nulls_after = int(df_clean["totalcharges"].isna().sum())
print(f"NULL totalcharges: {nulls_before} → {nulls_after} after imputation")

result_clean = validate_clean(df_clean, strict=False, reports_dir=REPORTS_DIR)

status_clean = "PASS" if result_clean.can_proceed else "FAIL"
print(f"Post-imputation validation: {status_clean}")
print(f"Errors: {len(result_clean.errors)}  Warnings: {len(result_clean.warnings)}")